# MahaMonsoon-X — Multiple Linear Regression
### Monthly gridded rainfall prediction • Processed ERA5 data

This notebook trains **one family of models only: regularized multiple linear regression (Ridge)**. It uses the existing `data/processed/era5_processed.csv`, constructs same-grid historical rainfall lags, tunes the regularization strength on a chronological validation period, evaluates on later held-out years, and saves all model and evaluation artifacts.

**Scope and caveats.** Each prediction represents one target month at one latitude/longitude grid cell. The CSV's target units are retained. If `tp` was standardized upstream using the whole dataset, this notebook cannot reverse that leakage; use a train-only preprocessing pipeline for a strict scientific evaluation. Verify the original ERA5 precipitation unit conversion before describing results as millimetres.

## 1. Mathematical formulation

For grid cell \(g\) and target month \(t\), let \(y_{t,g}\) be observed precipitation and \(\mathbf{x}_{t,g}\in\mathbb{R}^{p}\) be the predictor vector. Multiple linear regression estimates

\[
\widehat{y}_{t,g}=\beta_0+\sum_{j=1}^{p}\beta_j x_{t,g,j}.
\]

Here \(\beta_0\) is the intercept, \(\beta_j\) is the coefficient for feature \(j\), and \(p\) is the number of predictors.

Ordinary least squares minimizes the sum of squared residuals:

\[
\hat{\boldsymbol{\beta}}=
\underset{\boldsymbol{\beta}}{\operatorname{arg\,min}}
\sum_{i=1}^{n}(y_i-\beta_0-\mathbf{x}_i^\top\boldsymbol{\beta})^2.
\]

Weather predictors can be strongly correlated. Therefore, this notebook tunes Ridge regression, a regularized multiple linear model:

\[
\hat{\boldsymbol{\beta}}_\lambda=
\underset{\boldsymbol{\beta}}{\operatorname{arg\,min}}
\left[
\sum_{i=1}^{n}(y_i-\beta_0-\mathbf{x}_i^\top\boldsymbol{\beta})^2+
\lambda\sum_{j=1}^{p}\beta_j^2
\right].
\]

The intercept is not penalized. Larger \(\lambda\) shrinks coefficients more strongly. The validation period selects \(\lambda\); the final test years are not used for tuning.

### Evaluation metrics

\[
\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|,
\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2}.
\]

\[
R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.
\]

Lower MAE/RMSE is better. \(R^2=1\) is perfect, \(R^2=0\) matches the test-set mean baseline, and negative \(R^2\) is possible. Regression has no conventional classification accuracy, so this notebook reports MAE, RMSE, and \(R^2\), not an invented accuracy percentage.

## 2. Imports, paths, and reproducibility

Run from the project root. All outputs are isolated under `artifacts/linear_regression/`.

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().parent

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "linear_regression"

PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "era5_processed.csv"

MODEL_DIR = ARTIFACT_ROOT / "models"
RESULTS_DIR = ARTIFACT_ROOT / "results"
FIGURE_DIR = ARTIFACT_ROOT / "figures"
for folder in (MODEL_DIR, RESULTS_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found: {CSV_PATH.resolve()}. Run from the project root or update CSV_PATH."
    )
print("Input:", CSV_PATH.resolve())
print("Outputs:", ARTIFACT_ROOT.resolve())


## 3. Load the processed CSV

The CSV uses `timestamp`, `latitude`, and `longitude`; year/month are derived from `timestamp`. This cell does not redo upstream preprocessing.

In [ ]:
data = pd.read_csv(CSV_PATH)
data.columns = [str(c).strip() for c in data.columns]
data["timestamp"] = pd.to_datetime(data["timestamp"], errors="raise")
data["year"] = data["timestamp"].dt.year.astype(int)
data["month"] = data["timestamp"].dt.month.astype(int)

year_col, month_col = "year", "month"
lat_col, lon_col, target_col = "latitude", "longitude", "tp"
required = ["timestamp", year_col, month_col, lat_col, lon_col, target_col]
missing = [c for c in required if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in [lat_col, lon_col, target_col]:
    data[col] = pd.to_numeric(data[col], errors="raise")

excluded = {"timestamp", year_col, month_col, lat_col, lon_col, target_col}
weather_columns = [
    c for c in data.columns
    if c not in excluded and pd.api.types.is_numeric_dtype(data[c])
]
if not weather_columns:
    raise ValueError("No numeric weather predictors found.")

data = data.sort_values([lat_col, lon_col, "timestamp"]).reset_index(drop=True)
print("Data shape:", data.shape)
print("Date range:", data["timestamp"].min(), "to", data["timestamp"].max())
print("Target:", target_col)
print("Weather predictors:", weather_columns)
print("Grid cells:", data[[lat_col, lon_col]].drop_duplicates().shape[0])
display(data.head())


## 4. Create supervised features

Each row is a prediction for one month and one grid cell. Features include the weather predictors in the CSV, same-grid rainfall lags, and cyclical month encoding. The current `tp` target is never used as a predictor.

\[
s_m=\sin\left(\frac{2\pi m}{12}\right),\qquad
c_m=\cos\left(\frac{2\pi m}{12}\right).
\]

Sine/cosine encoding preserves the cyclical relationship between December and January. Rainfall lags capture persistence at the same grid cell.

In [ ]:
data = data.sort_values([lat_col, lon_col, "timestamp"]).copy()
target_group = data.groupby([lat_col, lon_col], sort=False)[target_col]
LAG_MONTHS = [1, 2, 3, 6, 12]
for lag in LAG_MONTHS:
    data[f"tp_lag_{lag}"] = target_group.shift(lag)

data["month_sin"] = np.sin(2 * np.pi * data[month_col] / 12.0)
data["month_cos"] = np.cos(2 * np.pi * data[month_col] / 12.0)
lag_columns = [f"tp_lag_{lag}" for lag in LAG_MONTHS]
feature_columns = weather_columns + lag_columns + ["month_sin", "month_cos"]

supervised = data.dropna(subset=["timestamp", target_col] + lag_columns).copy()
supervised = supervised.replace([np.inf, -np.inf], np.nan)
print("Supervised rows:", f"{len(supervised):,}")
print("Feature count:", len(feature_columns))
display(supervised[["timestamp", lat_col, lon_col, target_col] + lag_columns].head())


## 5. Chronological train / validation / test split

Split by target year so future years cannot enter the training set. All grid cells from the same target year remain in the same partition. This is a temporal holdout, not an independent spatial holdout; nearby grid cells remain spatially correlated.

In [ ]:
eligible_years = np.sort(supervised["year"].unique())
if len(eligible_years) < 5:
    raise ValueError(
        f"Only {len(eligible_years)} eligible years remain after creating lags; "
        "at least 5 are recommended."
    )
n_years = len(eligible_years)
n_train = max(1, int(n_years * 0.70))
n_val = max(1, int(n_years * 0.15))
if n_train + n_val >= n_years:
    n_train, n_val = n_years - 2, 1

train_years = eligible_years[:n_train]
val_years = eligible_years[n_train:n_train+n_val]
test_years = eligible_years[n_train+n_val:]

train_df = supervised[supervised["year"].isin(train_years)].copy()
val_df = supervised[supervised["year"].isin(val_years)].copy()
test_df = supervised[supervised["year"].isin(test_years)].copy()
print(f"Train: {train_years[0]}–{train_years[-1]} | {len(train_df):,} rows")
print(f"Validation: {val_years[0]}–{val_years[-1]} | {len(val_df):,} rows")
print(f"Test: {test_years[0]}–{test_years[-1]} | {len(test_df):,} rows")
if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError("A chronological split is empty.")


## 6. Tune Ridge regularization

Ridge is still a multiple linear regression model. Its L2 penalty helps stabilize coefficients when weather features are correlated. The pipeline fits any remaining predictor imputation and feature scaling on the training data only. Validation MAE selects the penalty parameter \(\lambda\).

In [ ]:
X_train, y_train = train_df[feature_columns].copy(), train_df[target_col].astype(float)
X_val, y_val = val_df[feature_columns].copy(), val_df[target_col].astype(float)
X_test, y_test = test_df[feature_columns].copy(), test_df[target_col].astype(float)

ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
tuning_rows = []
best_alpha, best_val_mae, best_pipeline = None, float("inf"), None

for alpha in ALPHAS:
    candidate = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    candidate.fit(X_train, y_train)
    pred = candidate.predict(X_val)
    mae = mean_absolute_error(y_val, pred)
    rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
    r2 = float(r2_score(y_val, pred)) if len(y_val) > 1 else np.nan
    tuning_rows.append({
        "alpha": alpha, "validation_MAE": mae,
        "validation_RMSE": rmse, "validation_R2": r2
    })
    print(f"alpha={alpha:g} | MAE={mae:.6f} | RMSE={rmse:.6f} | R2={r2:.4f}")
    if mae < best_val_mae:
        best_alpha, best_val_mae, best_pipeline = alpha, mae, candidate

tuning_results = pd.DataFrame(tuning_rows).sort_values("alpha")
display(tuning_results)
tuning_results.to_csv(RESULTS_DIR / "hyperparameter_tuning.csv", index=False)
print("Selected alpha:", best_alpha)


## 7. Refit using training + validation years

After choosing alpha, refit the pipeline on train and validation data. The final test years remain unseen until final evaluation.

In [ ]:
trainval_df = pd.concat([train_df, val_df], ignore_index=True)
lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=best_alpha))
])
lr_pipeline.fit(trainval_df[feature_columns], trainval_df[target_col].astype(float))
lr_predictions = lr_pipeline.predict(X_test)
print("Final multiple linear regression model fitted.")
print("Test predictions:", len(lr_predictions))


## 8. Final held-out test metrics

Metrics are reported in the target's current processed units. If `tp` was standardized upstream, these are standardized units, not millimetres.

In [ ]:
test_mae = mean_absolute_error(y_test, lr_predictions)
test_rmse = float(np.sqrt(mean_squared_error(y_test, lr_predictions)))
test_r2 = float(r2_score(y_test, lr_predictions)) if len(y_test) > 1 else np.nan
metrics = pd.DataFrame([{
    "model": "Ridge Multiple Linear Regression",
    "selected_alpha": best_alpha,
    "MAE": test_mae, "RMSE": test_rmse, "R2": test_r2,
    "test_rows": len(y_test),
    "test_year_start": int(test_years[0]),
    "test_year_end": int(test_years[-1])
}])
display(metrics.round(6))
metrics.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)

comparison = test_df[
    ["timestamp", "year", "month", lat_col, lon_col, target_col]
].copy().rename(columns={target_col: "actual"})
comparison["prediction"] = lr_predictions
comparison["residual_actual_minus_prediction"] = comparison["actual"] - comparison["prediction"]
comparison.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
display(comparison.head())


## 9. Diagnostic plots

The sample plot is a diagnostic sample across held-out grid-month observations, not a continuous time series for one location. Every figure is saved to the artifact folder.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.semilogx(tuning_results["alpha"], tuning_results["validation_MAE"], marker="o")
plt.xlabel("Ridge alpha (log scale)")
plt.ylabel("Validation MAE")
plt.title("Hyperparameter Tuning")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "validation_mae_by_alpha.png", dpi=160, bbox_inches="tight")
plt.show()

plot_sample = comparison.sample(n=min(2000, len(comparison)), random_state=SEED).sort_values(
    ["timestamp", lat_col, lon_col]
)
plt.figure(figsize=(13, 5))
plt.plot(plot_sample["actual"].to_numpy(), label="Actual", linewidth=1)
plt.plot(plot_sample["prediction"].to_numpy(), label="Linear regression prediction", linewidth=1)
plt.xlabel("Sampled held-out observations")
plt.ylabel("Processed tp units")
plt.title("Held-out Actual vs Predicted")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "test_prediction_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(comparison["actual"], comparison["prediction"], s=8, alpha=0.3)
low = float(min(comparison["actual"].min(), comparison["prediction"].min()))
high = float(max(comparison["actual"].max(), comparison["prediction"].max()))
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual processed tp")
plt.ylabel("Predicted processed tp")
plt.title("Actual vs Predicted")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "test_actual_vs_predicted.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4.5))
plt.hist(comparison["residual_actual_minus_prediction"], bins=50)
plt.axvline(0, linewidth=1)
plt.xlabel("Residual (actual − predicted)")
plt.ylabel("Count")
plt.title("Test Residual Distribution")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "test_residual_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].bar(["MAE"], [test_mae])
axes[0].set_title("Test MAE (lower is better)")
axes[1].bar(["RMSE"], [test_rmse])
axes[1].set_title("Test RMSE (lower is better)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "test_error_metrics.png", dpi=160, bbox_inches="tight")
plt.show()
print("Figures saved:", FIGURE_DIR.resolve())


## 10. Inspect coefficients

The predictor scaler means coefficients are expressed per standardized feature unit. Coefficients show fitted linear associations, not causation; correlated weather variables can share explanatory weight.

In [ ]:
ridge_model = lr_pipeline.named_steps["model"]
coef_table = pd.DataFrame({"feature": feature_columns, "coefficient": ridge_model.coef_})
coef_table["absolute_coefficient"] = coef_table["coefficient"].abs()
coef_table = coef_table.sort_values("absolute_coefficient", ascending=False)
display(coef_table.head(20))
coef_table.to_csv(RESULTS_DIR / "model_coefficients.csv", index=False)

top_coef = coef_table.head(20).sort_values("coefficient")
plt.figure(figsize=(9, 7))
plt.barh(top_coef["feature"], top_coef["coefficient"])
plt.xlabel("Ridge coefficient (standardized predictor)")
plt.title("Top 20 Coefficients by Absolute Magnitude")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "top_model_coefficients.png", dpi=160, bbox_inches="tight")
plt.show()


## 11. Save model and metadata

The saved pipeline includes the fitted imputer, feature scaler, and Ridge estimator. At inference time, construct exactly the same lag and calendar features in the same order before calling `predict`.

In [ ]:
model_path = MODEL_DIR / "multiple_linear_regression_pipeline.joblib"
metadata_path = MODEL_DIR / "model_metadata.json"
config_path = MODEL_DIR / "training_config.json"
joblib.dump(lr_pipeline, model_path)

metadata = {
    "project": "MahaMonsoon-X",
    "model": "Ridge regularized multiple linear regression",
    "input_csv": str(CSV_PATH),
    "target_column": target_col,
    "feature_columns": feature_columns,
    "weather_columns": weather_columns,
    "rainfall_lags_months": LAG_MONTHS,
    "calendar_features": ["month_sin", "month_cos"],
    "selected_alpha": float(best_alpha),
    "train_years": [int(train_years[0]), int(train_years[-1])],
    "validation_years": [int(val_years[0]), int(val_years[-1])],
    "test_years": [int(test_years[0]), int(test_years[-1])],
    "metrics": metrics.to_dict(orient="records"),
    "target_units_note": "Uses processed CSV target scale; verify upstream standardization and ERA5 units."
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
config_path.write_text(json.dumps({
    "seed": SEED,
    "alpha_grid": ALPHAS,
    "selected_alpha": float(best_alpha),
    "split_strategy": "chronological by target year",
    "refit_strategy": "training + validation years; test held out"
}, indent=2), encoding="utf-8")

for path in [model_path, metadata_path, config_path,
             RESULTS_DIR / "hyperparameter_tuning.csv",
             RESULTS_DIR / "test_metrics.csv",
             RESULTS_DIR / "test_predictions.csv",
             RESULTS_DIR / "model_coefficients.csv"]:
    print(f"Saved: {path.resolve()} ({path.stat().st_size / (1024**2):.2f} MiB)")


## 12. Reload and verify

The saved pipeline must reproduce the same predictions for the same features. This smoke test checks artifact integrity, not scientific model quality.

In [ ]:
loaded_pipeline = joblib.load(model_path)
n_check = min(100, len(X_test))
original = lr_pipeline.predict(X_test.iloc[:n_check])
reloaded = loaded_pipeline.predict(X_test.iloc[:n_check])
max_difference = float(np.max(np.abs(original - reloaded)))
assert np.allclose(original, reloaded, atol=1e-10, rtol=1e-8)

reload_test = X_test.iloc[:n_check].copy()
reload_test["actual"] = y_test.iloc[:n_check].to_numpy()
reload_test["prediction"] = reloaded
reload_test.to_csv(RESULTS_DIR / "reloaded_model_smoke_test.csv", index=False)
print("Reload verification passed. Maximum absolute difference:", max_difference)


## 13. Output inventory and interpretation checklist

```text
artifacts/linear_regression/
├── models/
│   ├── multiple_linear_regression_pipeline.joblib
│   ├── model_metadata.json
│   └── training_config.json
├── results/
│   ├── hyperparameter_tuning.csv
│   ├── test_metrics.csv
│   ├── test_predictions.csv
│   ├── model_coefficients.csv
│   └── reloaded_model_smoke_test.csv
└── figures/
    ├── validation_mae_by_alpha.png
    ├── test_prediction_comparison.png
    ├── test_actual_vs_predicted.png
    ├── test_residual_distribution.png
    ├── test_error_metrics.png
    └── top_model_coefficients.png
```

Before interpreting results, verify target units, upstream target standardization, and whether current-month predictors are available at the actual forecast issue time. This is a linear baseline and will not represent all nonlinear spatiotemporal rainfall dynamics. Syntax is checked when the notebook is generated; numerical results depend on the local CSV and installed package versions.